# NGSO SLS — Slice E: Live Spacetime Pull

**What this does:** Pulls the live NMTS network model + installed intents from a Spacetime/Minkowski instance, builds the constellation element array, runs the Slice-A H3 coverage engine on it, and compares predicted access against installed PathIntent routes.

**Read-only, Increment-1 scope:** no Create/Update/Delete. Keplerian motion fully supported; TLE/ephemeris flagged and skipped (Sgp4 deferred). NMTS→coverage path gated behind the **Increment-0 capability probe** (Cell 3).

---

> **SECURITY — NEVER PASTE KEYS INTO THIS NOTEBOOK.**
> Source all secrets from Colab secrets () or environment variables.
> Credentials must never appear as notebook literals, outputs, or git-tracked text.
> The PRIVATE_KEY_FILE field contains a local **path** only — upload the file via the Colab Files panel, never paste the key content.

In [ ]:
# === Install: spacetime-api (private index) + ngso_sls ===
# Run once per runtime; restart kernel after if prompted.

# 1. Live Spacetime API from the private Artifact Registry index
# shellcheck disable=SC2016
!pip install --upgrade spacetime-api     --extra-index-url https://us-central1-python.pkg.dev/a5a-spacetime-artifacts/py-packages/simple     --prefer-binary -q

# 2. ngso_sls — clone + non-editable install (mirrors 01_slice_a_mvp.ipynb)
REPO_URL = "https://github.com/luca-aalyria/spacetime-sls.git"

import importlib, importlib.util, subprocess, sys, os, re

def _run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("$", cmd)
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
        raise RuntimeError(f"command failed (exit {p.returncode}) — see output above")

if importlib.util.find_spec("ngso_sls") is None:
    url = REPO_URL
    try:
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
        if _tok and url.startswith("https://github.com/"):
            url = url.replace("https://", f"https://{_tok}@")
    except Exception:
        pass
    repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
    if not os.path.isdir(repo_dir):
        _run(f"git clone {url} {repo_dir}")
    else:
        _run(f"git -C {repo_dir} pull --ff-only")
    _run(f"{sys.executable} -m pip install {os.path.abspath(repo_dir)} -q")
    sys.path.insert(0, os.path.abspath(repo_dir))
    importlib.invalidate_caches()

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

In [ ]:
# === Increment-0 capability probe ===
# Checks which Spacetime API surfaces are available.
# HAS_MODEL=True  -> NMTS entity pull + coverage path runs.
# HAS_NBI=True    -> intents/routes pull runs.
# If any flag is False after installing spacetime-api, check the package version / index URL.

import ngso_sls.spacetime as st

flags = {k: getattr(st, k) for k in ["HAS_AUTH", "HAS_NBI", "HAS_PROVISIONING", "HAS_MODEL", "HAS_NMTS"]}
print("Capability flags:", flags)

if not flags["HAS_MODEL"]:
    print("
WARNING: HAS_MODEL=False — NMTS entity pull + coverage path will not run.")
    print("  The intents/provisioning demo still runs if HAS_NBI=True.")
    print("  Check: pip install spacetime-api (private index, see Cell 2).")
else:
    print("
HAS_MODEL=True — full NMTS → elements → coverage path available.")

In [ ]:
# === Connection form ===
# Sources secrets from Colab userdata where possible.
# NEVER hardcode a key here.  Upload the .key file via the Colab Files panel.

import os

def _secret(name, fallback=""):
    """Read from Colab userdata, then env, then return fallback (never raise)."""
    try:
        from google.colab import userdata
        return userdata.get(name) or fallback
    except Exception:
        return os.environ.get(name, fallback)

try:
    import ipywidgets as widgets
    from IPython.display import display

    w_url   = widgets.Text(value="https://fss01-demo.spacetime.aalyria.com:443",
                           description="URL:", layout=widgets.Layout(width="600px"))
    w_key   = widgets.Password(value=_secret("SPACETIME_KEY_ID"),
                               description="KEY_ID:", layout=widgets.Layout(width="400px"))
    w_user  = widgets.Password(value=_secret("SPACETIME_USER_ID"),
                               description="USER_ID:", layout=widgets.Layout(width="400px"))
    w_pkf   = widgets.Text(value=_secret("SPACETIME_KEY_FILE", "/content/spacetime.key"),
                           description="KEY_FILE path:", layout=widgets.Layout(width="500px"))
    w_murl  = widgets.Text(value="",
                           description="MODEL_URL:",
                           placeholder="Leave blank to use main URL",
                           layout=widgets.Layout(width="600px"))
    w_mver  = widgets.Dropdown(options=["v1", "v1alpha", "v0"], value="v1",
                               description="model_ver:")

    display(widgets.VBox([
        widgets.HTML("<b>Connection (secrets via Colab userdata — never paste keys here)</b>"),
        w_url, w_key, w_user, w_pkf, w_murl, w_mver,
    ]))

    def _build_endpoint():
        return st.SpacetimeEndpoint(
            url=w_url.value.strip(),
            key_id=w_key.value.strip(),
            user_id=w_user.value.strip(),
            private_key_file=w_pkf.value.strip(),
            model_url=w_murl.value.strip() or None,
            model_version=w_mver.value,
        )

    print("Fill in the form above, then run Cell 5.")

except ImportError:
    # Non-widget fallback (plain env / Colab userdata)
    _url  = _secret("SPACETIME_URL", "https://fss01-demo.spacetime.aalyria.com:443")
    _kid  = _secret("SPACETIME_KEY_ID")
    _uid  = _secret("SPACETIME_USER_ID")
    _pkf  = _secret("SPACETIME_KEY_FILE", "/content/spacetime.key")
    _murl = _secret("SPACETIME_MODEL_URL") or None
    _mver = _secret("SPACETIME_MODEL_VERSION", "v1")

    def _build_endpoint():
        return st.SpacetimeEndpoint(url=_url, key_id=_kid, user_id=_uid,
                                    private_key_file=_pkf, model_url=_murl, model_version=_mver)

    print(f"ipywidgets not available; using env/userdata. URL={_url}")

In [ ]:
# === First-contact SMOKE PROBE (run this BEFORE the full pull) ===
# Staged diagnostic — capability flags -> auth/channel -> intents -> entities -> platform
# motion -> relationships, one RPC at a time with PASS/FAIL, so first contact is debuggable.
# It never logs your key. Also runnable as a CLI:
#   SPACETIME_URL=... SPACETIME_KEY_ID=... SPACETIME_USER_ID=... \
#   SPACETIME_PRIVATE_KEY_FILE=/path/key python -m ngso_sls.spacetime.probe
from ngso_sls.spacetime.probe import probe_endpoint, format_report

report = probe_endpoint(_build_endpoint())
print(format_report(report))

# How to read it / common fixes:
#   * capability flags all False      -> spacetime-api not installed; re-run the install cell.
#   * live store NOT built            -> auth/URL/key issue (KEY_ID, USER_ID, key-file path, :443).
#   * [FAIL] intents (Unimplemented)  -> instance exposes a different NBI (legacy NetOps) surface.
#   * [FAIL] entities (unknown service model.v1.Model) -> set model_ver = v1alpha (or v0) in the
#                                        form above and re-run this cell.
#   * platform motion "0 keplerian served" -> instance stores TLE/ephemeris motion; those are
#                                        skipped in Increment-1 (Sgp4 is the deferred follow-up).
# Proceed to the full pull (next cells) once at least intents + entities PASS.

In [ ]:
# === Build live store ===
# Requires HAS_AUTH + HAS_MODEL (spacetime-api installed and probe passed).
# The store object is read-only: no Create/Update/Delete is ever wired.

from ngso_sls.spacetime.client import GrpcEntityStore

endpoint = _build_endpoint()
print(f"Connecting to: {endpoint.url}  (model_version={endpoint.model_version})")

store = GrpcEntityStore(endpoint)
print("GrpcEntityStore ready — pulling...")

In [ ]:
# === Pull model + build elements + run coverage ===
from datetime import datetime, timezone
from ngso_sls.spacetime.pull import pull_and_cover
from ngso_sls.grids.aor import AORS
import numpy as np

EPOCH_UTC  = datetime(2026, 1, 1, tzinfo=timezone.utc)
AOR        = AORS["India"]   # change to AORS["Global"] etc. as needed
CELL_RES   = 3               # H3 resolution (3=~100 km cells; 4=~30 km)
MIN_ELEV   = 25.0            # degrees
DURATION_S = 3600.0          # simulation window (s)
STEP_S     = 60.0            # time step (s)
K_VALUES   = (1, 2)          # k-coverage grades to compute

out = pull_and_cover(
    store, AOR,
    cell_res=CELL_RES,
    min_elev_user_deg=MIN_ELEV,
    duration_s=DURATION_S,
    step_s=STEP_S,
    epoch_utc=EPOCH_UTC,
    k_values=K_VALUES,
)

print(f"n_served  : {out["n_served"]} Keplerian satellites")
print(f"skipped   : {len(out["skipped"])} platforms (TLE / external-system)")
for s in out["skipped"]:
    print(f"  skip  sat_id={s["sat_id"]}  reason={s["reason"]}")

# Regularity summary
elems_a = np.array([m["epoch_utc_s"] for m in out["meta"]])
print(f"
Pulled {out["n_served"]} sats; ref_epoch={out["ref_epoch_s"]:.0f} s (UTC)")
cells   = out["coverage"]["cells"]
avail_k1 = out["coverage"]["availability"][:, 0] if out["coverage"]["availability"].ndim > 1 else out["coverage"]["availability"]
print(f"Coverage cells: {len(cells)} | mean k=1 availability: {avail_k1.mean():.3f}")
print(f"Installed routes (hops): {len(out["routes"])}")

In [ ]:
# === Coverage viz ===
from ngso_sls.viz.plots import plot_coverage_hexmap, plot_mbb_feasible_hexmap

print("k=1 coverage (availability):")
plot_coverage_hexmap(out["coverage"], k=1, title="Live Spacetime pull — k=1 coverage (India AOR)")

# MBB feasibility (only if continuity was requested)
if out["coverage"].get("mbb_feasible") is not None:
    print("MBB feasibility map:")
    plot_mbb_feasible_hexmap(out["coverage"], title="Live Spacetime pull — MBB feasibility (India AOR)")

In [ ]:
# === Compare predicted access vs installed routes ===
# For each installed PathIntent hop, check whether the SLS predicts the src satellite
# is in-view of the dst satellite during the simulation window, and write a validation report.
import csv, pathlib
from datetime import datetime, timezone

routes = out["routes"]
cells  = out["coverage"]["cells"]
avail  = out["coverage"]["availability"]

report_rows = []
for hop in routes:
    src = hop["src"]; dst = hop["dst"]
    src_if = hop.get("src_if"); dst_if = hop.get("dst_if")
    # Map source node to a meta entry (if available) for feasibility comment
    src_meta = next((m for m in out["meta"] if m["sat_id"] == src), None)
    dst_meta = next((m for m in out["meta"] if m["sat_id"] == dst), None)
    # Simple check: both nodes are served Keplerian satellites (not skipped)
    src_served = src_meta is not None
    dst_served = dst_meta is not None
    status = "BOTH_SERVED" if (src_served and dst_served) else (
             "SRC_SKIPPED" if not src_served else "DST_SKIPPED")
    report_rows.append({
        "src": src, "dst": dst, "src_if": src_if, "dst_if": dst_if,
        "src_served": src_served, "dst_served": dst_served, "status": status,
    })

# Write CSV
report_path = pathlib.Path("validation_report.csv")
with open(report_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["src","dst","src_if","dst_if",
                                            "src_served","dst_served","status"])
    writer.writeheader()
    writer.writerows(report_rows)

print(f"Validation report: {len(report_rows)} hops written to {report_path}")
for r in report_rows[:10]:
    print(f"  {r["src"]}->{r["dst"]}  {r["status"]}")
if len(report_rows) > 10:
    print(f"  ... and {len(report_rows)-10} more")

In [ ]:
# === Record live snapshot for offline replay ===
# Serialises the store's read surface to a proto-free JSON projection.
# Replay offline with: RecordedEntityStore("spacetime_live_snapshot.json")
# IMPORTANT: this file may contain sensitive topology — treat as confidential.

import ngso_sls.spacetime as st
import pathlib

snapshot_path = "spacetime_live_snapshot.json"
st.record(store, snapshot_path, intent_states=["INSTALLED"])
size_kb = pathlib.Path(snapshot_path).stat().st_size // 1024
print(f"Snapshot written: {snapshot_path}  ({size_kb} KB)")
print("Replay offline: from ngso_sls.spacetime.recording import RecordedEntityStore")
print(f"                rec = RecordedEntityStore("{snapshot_path}")")